<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1.What one row means: In our fact table, one row represents the performance of a single unique content page (URL) for a specific client on a single specific day.

2. Tables used: dim_content for page metadata, joined with fact_content_daily_performance for the time-series metrics.

3. Time window: I am iterating on a mid-panel month: March 2026 (2026-03). I am deliberately avoiding the _sample table (June 2026) to keep the final month sealed as a pure test set.

4. Target/Proxy: I am predicting is_declining_label (where trend_direction is "down").

5. Deliberately Excluded: I am completely excluding product-generated decision flags (like health_score, priority_score, or action_type) to ensure the model learns from raw evidence, not from the existing FlyRank system's rules.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

impressions_90d: Knowable at the decision moment because it represents historical search visibility exactly as recorded prior to today.

content_age_days: Knowable at the decision moment because the publication date is static metadata.

days_since_last_update: Knowable at the decision moment as it measures the time elapsed from the last historical edit.

avg_position: Knowable at the decision moment because it calculates historical average ranking position over the preceding window.

word_count: Knowable at the decision moment because it is the current, observable length of the text on the page.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
!pip install -U duckdb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 74.0 MB/s eta 0:00:00


In [1]:
import duckdb
from google.colab import userdata

print(duckdb.__version__)

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("--- FACT 1: Row Count and Date Span ---")
q1 = """
SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
display(con.execute(q1).df())

print("\n--- FACT 2: The Grain (Is it really Day x Client x Content?) ---")
q2 = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT report_date || client_hash_id || content_hash_id) as unique_grain_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
display(con.execute(q2).df())

print("\n--- FACT 3: Availability (Filtering with IS TRUE) ---")
q3 = """
SELECT COUNT(*) as rows_with_analytics
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE ga4_data_available IS TRUE
"""
display(con.execute(q3).df())

1.5.5
--- FACT 1: Row Count and Date Span ---


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31



--- FACT 2: The Grain (Is it really Day x Client x Content?) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_count
0,9841378,9841378



--- FACT 3: Availability (Filtering with IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_analytics
0,413966


The Leakage Trap Experiment: To demonstrate leakage, I intentionally added trend_pct to my feature set. Because our target label (is_declining_label) is mathematically derived from the trend calculation, trend_pct is simply the answer in disguise.

When I included it, the model's Precision@50 spiked to an unrealistic near-perfect score. The tree simply split on trend_pct < 0 and ignored all actual SEO signals. I have now deleted trend_pct from the feature frame to keep the model honest.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation of this slice: By filtering our analysis to rows where ga4_data_available IS TRUE, we are creating a blind spot. We are entirely dropping pages or clients that do not have Google Analytics tracking properly configured, meaning our model will not be able to score or prioritize a segment of the client's total web presence.